1. Shuffle tokenized sequences in test set
2. Inference original and shuffel input to the finetunned model 
3. find sequences which change classification from TN to FP after shuffling and save their indices for shap analysis
4. also find those remain TN after shuffling


In [2]:
import os
import pandas as pd
import numpy as np
# from pyfaidx import Fasta

import torch
import transformers

import matplotlib.pyplot as plt

%matplotlib inline


/p/project1/hai_dnaori/piroozeh1/venv_dnabert2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

data_dir="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb"

data = pd.read_csv(os.path.join(data_dir,"test_window.tsv"), sep='\t', header=0)
print(data)

     chr    start      end                                                seq  \
0     13   363312   363811  ATGCATTACAAATTTGTATTTGATTCTATATATAGTTTCGTTATAT...   
1      4   483669   484168  TTGATTTATACCATTTTGAAATCTAATCTTGCTCCAAGGGAGAAGC...   
2      7  1056150  1056649  CCTTGTAGTTGTCCTGCAAAGATGGGTAGGTATTTTCCTTGATATG...   
3     15   766442   766941  CAGTTTACAAGAATCACAACAAAAGGCGCCTTTGCAGAGGCCCCTT...   
4      4   638559   639058  CTCTAACTAATACAAACTCCTTACTATCTAATATAACGGCTACATC...   
..   ...      ...      ...                                                ...   
125   10    18014    18513  AGTAAAACTCTTGTGTCTTTTCATCGAACGTCCATGCAGAACCACC...   
126   16   698214   698713  TAAGCTCTGAGGGGCTTAATACTGTTGGTAAGACTCCAGAAATCGT...   
127   16   339219   339718  CTGGACCGCCCGGTAAGTTCTGCTATTAACAAATTAAATAAAATTG...   
128    4  1252944  1253443  AGAGGTCATTGTTCAAAGTGAGTTGGTGAGTTTTGGTCGGATACCC...   
129    8   555958   556457  ACATAACTGAATGTTAAAATGGCATCATCAATACAGTATTACTGAT...   

     label  
0        0  
1

In [4]:
# # shuffle_sequences.py

# from __future__ import annotations

# from pathlib import Path
# from typing import Union

# import pandas as pd
# import torch
# import transformers



# def shuffling(
#     model_path: str,
#     data: pd.DataFrame,
#     out_csv_path
# ) -> pd.DataFrame:
#     """
#     Read sequences+labels from a DataFrame, shuffle tokens for each sequence
#     (keeping [CLS]/[SEP] fixed; ignoring [PAD]), decode back to text, and
#     save to CSV with columns: sequence, label.

#     Notes
#     -----
#     - Shuffling happens at tokenizer-token level (not character-level).
#     - Output sequences include special tokens only if your tokenizer decode keeps them;
#       we decode with skip_special_tokens=True by default for cleaner text.
#     """

#     out_csv_path = Path(out_csv_path)

#     tokenizer = transformers.AutoTokenizer.from_pretrained(
#         model_path,
#         model_max_length=125,
#         padding_side="right",
#         use_fast=True,
#         trust_remote_code=True,
#     )


#     pad_id = tokenizer.pad_token_id
#     cls_id = tokenizer.cls_token_id
#     sep_id = tokenizer.sep_token_id
#     if cls_id is None or sep_id is None:
#         raise ValueError("Tokenizer must define cls_token_id and sep_token_id for CLS/SEP shuffling.")

#     shuffled_sequences: list[str] = []
#     labels: list[int] = []

#     for _, row in data.iterrows():
#         # print(row)
#         seq = row["seq"]
#         lab = row["label"]

#         inputs = tokenizer(
#             str(seq),
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             max_length=125,
#         )
#         input_ids = inputs["input_ids"][0]  # (L,)
#         attention_mask = inputs["attention_mask"][0]  # (L,)

#         valid_len = int(attention_mask.sum().item())
#         shuffled_ids = input_ids.clone()

#         if valid_len > 2:
#             # Shuffle tokens between CLS (pos 0) and SEP (pos valid_len-1)
#             positions = torch.arange(1, valid_len - 1)
#             if positions.numel() > 1:
#                 perm = positions[torch.randperm(positions.numel())]
#                 shuffled_ids[positions] = shuffled_ids[perm]

#         # Decode only valid (non-pad) region; keep it clean by skipping specials
#         decoded = tokenizer.decode(
#             shuffled_ids[:valid_len].tolist(),
#             skip_special_tokens=True,
#             clean_up_tokenization_spaces=True,
#         ).strip()

#         shuffled_sequences.append("".join(decoded.split()))
#         labels.append(int(lab))

#     out_df = pd.DataFrame({"sequence": shuffled_sequences, "label": labels})
#     out_df.to_csv(out_csv_path, index=False)
#     return out_df


# # Example usage:
# # model_path = "/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/model2"
# # out_csv_path="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/shuffled_sequences.csv"
# # out_df = shuffling(model_path, data, out_csv_path)

In [5]:
pred_results  = np.load("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/pred_results_test_model2.npy")
data['pred'] = pred_results
data['pred'] = data['pred'].apply(lambda x: 1 if x>0.5 else 0)
print(pred_results)

[0.00596435 0.99383545 0.00848924 0.9945714  0.00856034 0.00591839
 0.9946089  0.9946149  0.9944834  0.00599306 0.9945393  0.9926347
 0.9945397  0.9945517  0.99462044 0.9945884  0.99456906 0.00586831
 0.99453133 0.00584184 0.9941666  0.9945457  0.9945364  0.99451506
 0.00751891 0.02401546 0.9921163  0.9945427  0.99459165 0.9945971
 0.00588414 0.9943072  0.00585898 0.99458706 0.9945793  0.00605409
 0.00593577 0.994586   0.99459547 0.00590151 0.994557   0.99456257
 0.9945298  0.00644075 0.00626989 0.00599648 0.994494   0.994592
 0.9943626  0.00592392 0.00575113 0.0059236  0.00830391 0.9945741
 0.9944653  0.99455726 0.9945622  0.9945949  0.9933296  0.99314004
 0.00593962 0.9946163  0.9944711  0.00613169 0.9945779  0.00716066
 0.99452716 0.9944194  0.00582923 0.99454355 0.97786385 0.99454826
 0.00599439 0.9945585  0.9945109  0.01332614 0.99384874 0.99457294
 0.9945076  0.9945773  0.9941459  0.9945809  0.00617142 0.9940228
 0.9946038  0.00590487 0.99452966 0.911796   0.99452704 0.9945634
 0

In [7]:
# save TP and TN indeces
matched_indices = []

for index, row in data.iterrows():
    if(row['label']==0 and row['pred']==0):
        matched_indices.append(index)
        print(index)
pd.Series(matched_indices, name="index").to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/TN_indices.csv", index=False)        

0
2
4
5
9
17
19
24
25
30
32
35
36
39
43
44
45
49
50
51
52
60
63
65
68
72
75
85
90
91
96
99
101
106
107
108
109
113
117
118
123
124
125
128


In [8]:
# Ensure integer type
data["label"] = data["label"].astype(int)
data["pred"] = data["pred"].astype(int)

tp = ((data["label"] == 1) & (data["pred"] == 1)).sum()
tn = ((data["label"] == 0) & (data["pred"] == 0)).sum()
fp = ((data["label"] == 0) & (data["pred"] == 1)).sum()
fn = ((data["label"] == 1) & (data["pred"] == 0)).sum()

print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")

TP: 61
TN: 44
FP: 21
FN: 4


In [5]:
def new_inference_model( model_path, sequence):
    model_path = model_path
    config = transformers.BertConfig.from_pretrained(
                model_path,
                num_labels=2
        )
    config.output_attentions = True
    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path, config=config, from_tf=False,trust_remote_code=True)
    
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        model_path,
        model_max_length=125,
        padding_side="right",
        use_fast=True,
        trust_remote_code=True
        
    )

   
    
    shuffled_sequences: str = ""
    inputs = tokenizer(sequence, return_tensors="pt")
  
    input_ids = inputs['input_ids']

    #perturbing input by shuffling tokens, except CLS, SEP
    fixed_indices = (input_ids == 1) | (input_ids == 2)

    # Extract tokens to shuffle (non-fixed)
    tokens_to_shuffle = input_ids[~fixed_indices]
    shuffled_indices=torch.randperm(tokens_to_shuffle.size(0))
    # Shuffle tokens
    # print(tokens_to_shuffle)
    shuffled_tokens = tokens_to_shuffle[shuffled_indices]
    # print(shuffled_tokens)
    # Recover original positions from shuffled indices
    inverse_indices = torch.argsort(shuffled_indices)
    # Create shuffled output
    shuffled_input_ids = input_ids.clone()
    shuffled_input_ids[~fixed_indices] = shuffled_tokens
    output2 = model(shuffled_input_ids)
   
    logits2 = output2.logits
    pred2=logits2.detach().numpy()
  
    softmax = torch.nn.Softmax(dim=1)
    prob2=softmax(torch.tensor(pred2)).numpy()
    #
    les_seq= len(shuffled_input_ids[0])
  
    decoded = tokenizer.decode(
            shuffled_input_ids[0][:les_seq].tolist(),
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        ).strip()

    shuffled_sequences=("".join(decoded.split()))   
    
    return prob2, shuffled_sequences


In [6]:
model_path = "/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/model2"
 
new_preds = []
shuf_seqs = []
for index, row in data.iterrows():
    # if(index <2):
        prob, shuf_seq = new_inference_model(model_path, row["seq"])  # shape (1, 2)
        shuf_seqs.append(shuf_seq)
        pred = 1 if prob[0, 1] > 0.50 else 0
        new_preds.append(pred)

data["new_pred"] = np.array(new_preds, dtype=np.int64)
data["shuf_seq"] = shuf_seqs

print(data)


/p/software/juwels/stages/2024/software/PyTorch/2.1.2-gcccoreflexiblas-12.3.0-3.3.1/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/p/home/jusers/piroozeh1/juwels/.cache/huggingface/modules/transformers_modules/model2/bert_layers.py:128: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(


     chr    start      end                                                seq  \
0     13   363312   363811  ATGCATTACAAATTTGTATTTGATTCTATATATAGTTTCGTTATAT...   
1      4   483669   484168  TTGATTTATACCATTTTGAAATCTAATCTTGCTCCAAGGGAGAAGC...   
2      7  1056150  1056649  CCTTGTAGTTGTCCTGCAAAGATGGGTAGGTATTTTCCTTGATATG...   
3     15   766442   766941  CAGTTTACAAGAATCACAACAAAAGGCGCCTTTGCAGAGGCCCCTT...   
4      4   638559   639058  CTCTAACTAATACAAACTCCTTACTATCTAATATAACGGCTACATC...   
..   ...      ...      ...                                                ...   
125   10    18014    18513  AGTAAAACTCTTGTGTCTTTTCATCGAACGTCCATGCAGAACCACC...   
126   16   698214   698713  TAAGCTCTGAGGGGCTTAATACTGTTGGTAAGACTCCAGAAATCGT...   
127   16   339219   339718  CTGGACCGCCCGGTAAGTTCTGCTATTAACAAATTAAATAAAATTG...   
128    4  1252944  1253443  AGAGGTCATTGTTCAAAGTGAGTTGGTGAGTTTTGGTCGGATACCC...   
129    8   555958   556457  ACATAACTGAATGTTAAAATGGCATCATCAATACAGTATTACTGAT...   

     label  pred  new_pred 

In [ ]:
shuffled_sequences= data["shuf_seq"]
labels=data["label"]

out_df = pd.DataFrame({"seq": shuffled_sequences, "label": labels})
print(out_df)
out_csv_path="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/test_shuffled.tsv"
out_df.to_csv(out_csv_path, sep="\t", index=False)
# out_csv_path="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/test_all.tsv"
# out_df.to_csv(out_csv_path, sep="\t", index=False)

                                                   seq  label
0    TGATTAGGTTTTTTTGGATTTGGATATATGACCCATCAACGCCAAA...      0
1    GAGCCTCTTTTTTACCAGGAATGCCATATGAAAGCTACACCTTGGG...      1
2    TTTTAATGAGTGGGTGTTACTCAAATTACTATGAATTTGACAGAGA...      0
3    GTAGAAGTAAAACTGAGAATCAGATATATAATGCGAGATTAGGCGC...      1
4    GTAATAATAATAATACGCCAATAGCAACCTTGCATCTACGTCTAAC...      0
..                                                 ...    ...
125  GCGAGACCGTACATGCTGTTTAACCGTAAAAGTGGCCCCATTCAAA...      0
126  TCCCTTTCAAAAGATATTTTACTGCAGAAAAATACTTAGAAAGCCT...      0
127  AAATCGATGGCGGCAAAAAGGATTCTCACTAGCAATCCAAGGCCAA...      0
128  GGGAAAGCCAAGTGAGTTCAAACGCTAGTGCTAAGAGGGAAAAAGA...      0
129  TCCCATAAGATGGATTTGGGTATTCTTATGTTTACCGTGCCCTTTT...      1

[130 rows x 2 columns]


In [13]:
matched_indices = []

for index, row in data.iterrows():
    if(row['label']==0 and row['pred']==0 and row['new_pred']==1):
        matched_indices.append(index)
        print(index)
        print("seq:")
        print(row['seq'])
        print("Shuffeld seq:")
        print(row['shuf_seq'])
pd.Series(matched_indices, name="index").to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/TN_turnto_FP_indices.csv", index=False)        

0
seq:
ATGCATTACAAATTTGTATTTGATTCTATATATAGTTTCGTTATATAATAACTCAGGTCTGCTCTGCAGCGTGGTTTACGATAAACACGTATGTACCACTGTCTGCATCATAGCTTTCGAACTTAGAATTTGGATTTTTCTTTAAGCGTTCGATATGCCTTTTAACCAGTTGATGATTTGGATCCTTTATAGGTTTTCTGGTAGATTTGTCTACAGGATAACAGTTAAAGCAGGTAATCCTGGCCCGAACATTAATGCCCTCCCCTCTCTTCGGTCTGTTTGGTAAGTTCGCATATATTATACAGGTTTTTGGTTCAAAAGTAATGATTACGCCCCCTAATGAGGTCAAAGGAATTCCAGCGAGGTCAACAGGTTCAAGAAATTCAATTTTACCATAACTTTTATGACCCACAACTAAATGAGGAACTTTGCGCAGTTGCAAGAGAGAATAAGAGGATAATGTGTCCAGGGATGGTGAGATATAGTAGTTCTCATTAATG
Shuffeld seq:
TGATTAGGTTTTTTTGGATTTGGATATATGACCCATCAACGCCAAATCCTGTCAAATTTTAATATATATTTTTTTGGTTCCAGTTATGCCTCTGTTTGTGAGGAACTAAACGACATTAAGCGTGTGATTGAGATAATTAATGTTATAGCAGGTTACAAAGTGAGTTGAATAACAATAACTCATCGAACTATATAGTAATCCAGGGTAGTTGATGCGGCGCACCACTGCAGGTTTATAAGATTTTGCCTCTACACTCACAGTTGGAATTGCGAGGGCGTTCTCTGCACAGGCAAAAGTTTACCATAATCTGGAGGACTCCCGCATGCATAATGTGCGAAGAATTTGTTTGCTTAGAAATTTGAGGCCATATGCTGGTAGTTGCAACGAGGTCTGTCTTTAACTTTTACAACGTACAATTTTATAAACAGGATAACCCTAAGATGACATCAGATCCTTCCTCTCTTTATATAGCTTCCTC

In [14]:
matched_indices = []
for index, row in data.iterrows():
    if(row['label']==0 and row['pred']==0 and row['new_pred']==0):
        matched_indices.append(index)
        print(index)
        print("seq:")
        print(row['seq'])
        print("Shuffeld seq:")
        print(row['shuf_seq'])
pd.Series(matched_indices, name="index").to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/TN_remain_TN_indices.csv", index=False)             

2
seq:
CCTTGTAGTTGTCCTGCAAAGATGGGTAGGTATTTTCCTTGATATGGTTGCTATCGCATTTGCACTAATTATTACGTTATTGTGTGTTACGAGAGCCTTTCCTATTTCCGCGGCTTCAGTTGGTGTTTTGTTGACTTATGTATTACAATTGCCTGGTCTATTAAATACCATTTTAAGGGCAATGACTCAAACAGAGAATGACATGAATAGTGCCGAAAGATTGGTAACATATGCAACTGAACTACCACTAGAGGCATCCTATAGAAAGCCCGAAATGACACCTCCAGAGTCATGGCCCTCAATGGGCGAAATAATTTTTGAAAATGTTGATTTTGCCTATAGACCTGGTTTACCTATAGTTTTAAAAAATCTTAACTTGAATATCAAGAGTGGGGAAAAAATTGGTATCTGTGGTCGTACAGGTGCTGGTAAGTCCACTATTATGAGTGCCCTTTACAGGTTGAATGAATTGACCGCAGGTAAAATTTTAATTGACAATG
Shuffeld seq:
TTTTAATGAGTGGGTGTTACTCAAATTACTATGAATTTGACAGAGAATTTGCCGAAAGTCTATTAAAGAGTGGAATACAGTTCCACTACATATGGTAGTTAAAATTTACCCAATTCTTCTCAATCTGTGGTTGACTTTAAAAGATATGGTCGCGGTGAATATAGGTAAGACCTTTTGATGTGTATTAGAACACTAGACCTGTATTTTGAAACGAAATGACAGAGGGCCCACTAATTTTTGGTAAGTTGATTTTTCCGTCCTGGCTAGTCATGTTATTATGCTGTACCATCAATAAGGTAGTGTGGCCTTCTTAGCCTATACCCTTGGGCAATATACGAGACGAAAGGTAGGGCCTGTGTTCATCCGTACAGGCACCTATTTTAATTGTTGACAATGGATTGTCTAGTGGCCGCAGGCGTTATTCTGAAGTTTTTATTTCCTCCAGAAAAAAACAATGACCTTTCTTAATGGGTCGCAG

In [15]:
# Ensure integer type
data["label"] = data["label"].astype(int)
data["new_pred"] = data["new_pred"].astype(int)

tp = ((data["label"] == 1) & (data["new_pred"] == 1)).sum()
tn = ((data["label"] == 0) & (data["new_pred"] == 0)).sum()
fp = ((data["label"] == 0) & (data["new_pred"] == 1)).sum()
fn = ((data["label"] == 1) & (data["new_pred"] == 0)).sum()

print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")

TP: 65
TN: 32
FP: 33
FN: 0
